# LeetCode 371 — Sum of Two Integers bằng Python

Link đề: https://leetcode.com/problems/sum-of-two-integers/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation (thao tác bit), Math (toán)

Skeleton LeetCode:

```python
class Solution:
    def getSum(self, a: int, b: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ. Điểm khó riêng của Python: số nguyên **không giới hạn độ rộng**, nên ta phải **tự mô phỏng số nguyên 32 bit có dấu** (mục 3 và mục 7).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai số nguyên `a` và `b`, trả về **tổng** của chúng **mà không dùng toán tử `+` và `-`**.
2. Nắm các công cụ Python về bit: `&` (AND), `|` (OR), `^` (XOR), `<<` / `>>` (dịch bit), `~` (đảo bit), **mặt nạ (mask) `0xFFFFFFFF`**, `bin(x)` và `format(x, "032b")`.
3. Hiểu vì sao **Python khác C++/Java**: số nguyên của Python có độ rộng tùy ý, số âm là "vô hạn bit 1 bên trái", nên phải **mô phỏng số nguyên 32 bit có dấu** bằng mask và **đổi ngược** kết quả về số có dấu.
4. Đi từ cách **đếm tăng/giảm từng đơn vị bằng bit** → **cộng từng bit với nhớ (full adder)** → **XOR + AND dịch trái lặp đến khi hết nhớ** (cách nộp) → **tách dấu rồi cộng/trừ độ lớn** (không cần mask).
5. Tránh các lỗi kinh điển: **vòng lặp vô hạn** khi quên mask, **không đổi ngược** `4294967295` về `-1`, nhầm **thứ tự ưu tiên toán tử** (`a & b << 1`), và quên rằng `a ^ b` **chưa** tính phần nhớ.

## 2. Đề bài & ví dụ

Cho hai số nguyên `a` và `b`. Hãy trả về **tổng** của hai số **mà không được dùng toán tử `+` và `-`**.

**Ví dụ 1**

- Input: `a = 1`, `b = 2`
- Output: `3`

**Ví dụ 2**

- Input: `a = 2`, `b = 3`
- Output: `5`

**Ví dụ tự dựng (đối chiếu lại đề gốc)**

- `a = -1`, `b = 1` → `0`.
- `a = -5`, `b = -7` → `-12`.
- `a = 1000`, `b = -1000` → `0`.

**Ràng buộc**

- `-1000 <= a, b <= 1000`.

Đề hỏi một điều "hiển nhiên" (`a + b`), nhưng **cấm** dùng phép cộng và trừ. Vì vậy ta phải tự dựng phép cộng từ các phép toán **bit** — đúng cách mạch điện tử của máy tính cộng hai số. Bài này cũng là cách tốt để hiểu **số âm được lưu thế nào** trong máy.

In [1]:
import random
import itertools

examples = [
    (1, 2, 3),
    (2, 3, 5),
    (-1, 1, 0),
    (-5, -7, -12),
    (1000, -1000, 0),
]
for a, b, expected in examples:
    print(f"a={a}, b={b} -> kỳ vọng {expected}")

a=1, b=2 -> kỳ vọng 3
a=2, b=3 -> kỳ vọng 5
a=-1, b=1 -> kỳ vọng 0
a=-5, b=-7 -> kỳ vọng -12
a=1000, b=-1000 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Số nhị phân và các toán tử bit

Máy tính lưu số bằng **bit** (`0` hoặc `1`). Các toán tử bit làm việc **trên từng cột bit** riêng lẻ:

| Toán tử | Tên | Quy tắc trên một cột bit |
|---|---|---|
| `a & b` | AND | `1` chỉ khi **cả hai** đều `1` |
| `a \| b` | OR | `1` khi **ít nhất một** là `1` |
| `a ^ b` | XOR | `1` khi hai bit **khác nhau** |
| `a << k` | dịch trái | thêm `k` bit `0` bên phải (nhân `2^k`) |
| `a >> k` | dịch phải | bỏ `k` bit bên phải (chia `2^k`, làm tròn xuống) |
| `~a` | đảo bit | đổi `0 <-> 1` (với số nguyên Python: `~a` bằng `-a - 1`) |

In [2]:
a, b = 5, 3
print(f"a     = {a:04b}")
print(f"b     = {b:04b}")
print(f"a & b = {a & b:04b}   ({a & b})")
print(f"a | b = {a | b:04b}   ({a | b})")
print(f"a ^ b = {a ^ b:04b}   ({a ^ b})")
print(f"a << 1 = {a << 1:04b}  ({a << 1})")
print(f"a >> 1 = {a >> 1:04b}  ({a >> 1})")
print("~5 =", ~5, "(bằng -5 - 1)")

a     = 0101
b     = 0011
a & b = 0001   (1)
a | b = 0111   (7)
a ^ b = 0110   (6)
a << 1 = 1010  (10)
a >> 1 = 0010  (2)
~5 = -6 (bằng -5 - 1)


### 3.2. Cộng một cột bit: XOR là tổng, AND là nhớ

Hãy cộng hai **bit** `x` và `y` (mỗi bit `0` hoặc `1`). Chỉ có bốn khả năng:

| `x` | `y` | `x + y` | bit tổng (viết xuống) | bit nhớ (sang cột trái) |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 1 | 1 | 0 |
| 1 | 0 | 1 | 1 | 0 |
| 1 | 1 | 10 (nhị phân) | 0 | 1 |

Nhìn cột "bit tổng": đó đúng là **`x ^ y`**. Nhìn cột "bit nhớ": đó đúng là **`x & y`**. Vậy **XOR là tổng không tính nhớ**, **AND là các vị trí sinh ra nhớ**, và nhớ phải đi sang cột **bên trái** ⇒ dịch trái một bit: `(x & y) << 1`.

In [3]:
for x, y in itertools.product([0, 1], repeat=2):
    print(f"x={x} y={y}: x ^ y = {x ^ y}, x & y = {x & y}  -> x + y = {2 * (x & y) + (x ^ y)}")

x=0 y=0: x ^ y = 0, x & y = 0  -> x + y = 0
x=0 y=1: x ^ y = 1, x & y = 0  -> x + y = 1
x=1 y=0: x ^ y = 1, x & y = 0  -> x + y = 1
x=1 y=1: x ^ y = 0, x & y = 1  -> x + y = 2


### 3.3. Python khác C++/Java: số nguyên không có độ rộng cố định

Trong C++/Java, `int` luôn **32 bit**; số âm dùng **bù hai (two's complement)**: `-1` là `111...1` (32 chữ số `1`). Khi cộng bị tràn, bit thừa bị **bỏ đi** tự động.

Python 3 không như vậy: `int` có thể to tùy ý, và số âm hoạt động như thể có **vô hạn bit `1` bên trái**. Vì thế `-1 & 0xFFFFFFFF` cho `4294967295` (chính là 32 bit `1`) chứ không phải `-1`. **Ta phải tự "cắt" về 32 bit** bằng cách AND với mask `0xFFFFFFFF`.

In [4]:
MASK = 0xFFFFFFFF            # 32 bit 1: 1111 1111 1111 1111 1111 1111 1111 1111
print(MASK, bin(MASK), len(bin(MASK)) - 2, "bit")
print("-1 & MASK  =", -1 & MASK, format(-1 & MASK, "032_b"))
print("-2 & MASK  =", -2 & MASK, format(-2 & MASK, "032_b"))
print(" 5 & MASK  =", 5 & MASK, format(5 & MASK, "032_b"))
print("-1 >> 5    =", -1 >> 5, "(số âm dịch phải vẫn là -1: bit dấu được nhân bản)")

4294967295 0b11111111111111111111111111111111 32 bit
-1 & MASK  = 4294967295 1111_1111_1111_1111_1111_1111_1111_1111
-2 & MASK  = 4294967294 1111_1111_1111_1111_1111_1111_1111_1110
 5 & MASK  = 5 00_0000_0000_0000_0000_0000_0101
-1 >> 5    = -1 (số âm dịch phải vẫn là -1: bit dấu được nhân bản)


### 3.4. Đổi 32 bit không dấu về số có dấu

Sau khi cắt bằng mask, kết quả `r` nằm trong `[0, 2^32 - 1]`. Theo quy ước bù hai:

- Nếu `r <= 0x7FFFFFFF` (`2^31 - 1`): bit cao nhất là `0` ⇒ là **số không âm**, giữ nguyên.
- Nếu `r > 0x7FFFFFFF`: bit cao nhất là `1` ⇒ là **số âm** có giá trị thật `r - 2^32`. Mà ta không được dùng `-`, nên dùng công thức tương đương **`~(r ^ MASK)`**: `r ^ MASK` đảo 32 bit thấp (bằng `2^32 - 1 - r`), rồi `~` đảo toàn bộ (ra `-(2^32 - 1 - r) - 1 = r - 2^32`).

In [5]:
MASK = 0xFFFFFFFF
MAX_INT = 0x7FFFFFFF         # 2^31 - 1


def to_signed(r):
    """Đổi một số 32 bit không dấu (0 .. 2^32-1) về số nguyên có dấu của Python."""
    r &= MASK
    return r if r <= MAX_INT else ~(r ^ MASK)


for r in [0, 5, MAX_INT, MAX_INT + 1, 4294967294, 4294967295]:
    print(r, "->", to_signed(r), "| r - 2^32 =", r - 2 ** 32 if r > MAX_INT else r)

0 -> 0 | r - 2^32 = 0
5 -> 5 | r - 2^32 = 5
2147483647 -> 2147483647 | r - 2^32 = 2147483647
2147483648 -> -2147483648 | r - 2^32 = -2147483648
4294967294 -> -2 | r - 2^32 = -2
4294967295 -> -1 | r - 2^32 = -1


### 3.5. Hàm `abs`, `range` và `while` với điều kiện số

`abs(x)` trả giá trị tuyệt đối (không cần dấu `-` trong code của ta). `range(n)` lặp `n` lần. `while b:` lặp khi `b != 0` (số `0` là "sai"). Ba công cụ này ta dùng liên tục dưới đây.

In [6]:
print(abs(-7), abs(7), abs(0))
print(list(range(abs(-3))))
b = 3
count = 0
while b:                      # b khác 0 thì còn lặp
    b >>= 1                   # dịch phải: 3 -> 1 -> 0
    count += 1
print("số lần lặp:", count)

7 7 0
[0, 1, 2]
số lần lặp: 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cộng `5 + 3` trên giấy theo **nhị phân** (ta đặt `5 = 0101`, `3 = 0011`):

```
  0101   (5)
+ 0011   (3)
------
```

Từ phải sang trái: cột 0: `1+1 = 10` → viết `0` nhớ `1`; cột 1: `0+1+nhớ 1 = 10` → viết `0` nhớ `1`; cột 2: `1+0+nhớ 1 = 10` → viết `0` nhớ `1`; cột 3: `0+0+nhớ 1 = 1` → viết `1`. Kết quả `1000 = 8`.

Cách **làm song song cả hàng** (không từng cột): tính **tổng không nhớ** và **các bit nhớ** một lượt:

| Bước | `a` | `b` | `a ^ b` (tổng không nhớ) | `(a & b) << 1` (nhớ) |
|---|---|---|---|---|
| 0 | `0101` | `0011` | `0110` | `0010` |
| 1 | `0110` | `0010` | `0100` | `0100` |
| 2 | `0100` | `0100` | `0000` | `1000` |
| 3 | `0000` | `1000` | `1000` | `0000` |

Mỗi bước thay `(a, b)` bằng `(a ^ b, (a & b) << 1)` — **tổng `a + b` không đổi** nhưng phần "nhớ" `b` bị **đẩy dần sang trái** cho tới khi `b = 0`. Khi đó `a` chính là tổng: `1000 = 8`.

In [7]:
a, b = 5, 3
step = 0
while b:
    total_no_carry = a ^ b
    carry = (a & b) << 1
    print(f"bước {step}: a={a:04b} b={b:04b} | a^b={total_no_carry:04b} carry={carry:04b}")
    a, b = total_no_carry, carry
    step += 1
print("kết quả:", a, f"({a:04b})")

bước 0: a=0101 b=0011 | a^b=0110 carry=0010
bước 1: a=0110 b=0010 | a^b=0100 carry=0100
bước 2: a=0100 b=0100 | a^b=0000 carry=1000
bước 3: a=0000 b=1000 | a^b=1000 carry=0000
kết quả: 8 (1000)


## 5. Cách 1 — Brute Force / baseline: tăng hoặc giảm từng đơn vị bằng bit

**Ý tưởng.** Cộng `b` vào `a` nghĩa là **tăng `a` thêm `1`, `b` lần** (nếu `b > 0`) hoặc **giảm `a` đi `1`, `|b|` lần** (nếu `b < 0`). Ta không dùng `+ 1` được, nên tự viết hai hàm **bằng bit**:

- **Tăng 1 (`inc`)**: một chuỗi bit `1` thấp nhất bị lật thành `0`, và bit `0` đầu tiên ngay sau chuỗi đó thành `1`. Ví dụ `0111 + 1 = 1000`.
- **Giảm 1 (`dec`)**: một chuỗi bit `0` thấp nhất bị lật thành `1`, và bit `1` đầu tiên sau chuỗi đó thành `0`. Ví dụ `1000 - 1 = 0111`.

Làm việc với số **32 bit không dấu** (`x & MASK`), cuối cùng đổi về có dấu bằng `to_signed`.

In [8]:
def inc32(x):
    """Tăng x (32 bit không dấu) thêm 1 bằng thao tác bit."""
    m = 1
    while x & m:              # các bit 1 liên tiếp từ thấp nên bị lật thành 0
        x ^= m
        m <<= 1
    return (x ^ m) & MASK     # bit 0 đầu tiên được lật thành 1 (nếu tràn 32 bit thì bị cắt = 0)


def dec32(x):
    """Giảm x (32 bit không dấu) đi 1 bằng thao tác bit."""
    m = 1
    while m <= MASK and not (x & m):   # các bit 0 liên tiếp từ thấp nên bị lật thành 1
        x ^= m
        m <<= 1
    return (x ^ m) & MASK     # bit 1 đầu tiên được lật thành 0


print(f"{7:04b} inc -> {inc32(7):04b} | {8:04b} dec -> {dec32(8):04b} | 0 dec -> {dec32(0)} | MASK inc -> {inc32(MASK)}")


def get_sum_count(a, b):
    x = a & MASK
    for _ in range(abs(b)):                 # lặp |b| lần
        x = inc32(x) if b > 0 else dec32(x)
    return to_signed(x)


for a, b, expected in examples:
    print(get_sum_count(a, b), "(kỳ vọng", expected, ")")

0111 inc -> 1000 | 1000 dec -> 0111 | 0 dec -> 4294967295 | MASK inc -> 0
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )
0 (kỳ vọng 0 )
-12 (kỳ vọng -12 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Mỗi lần tăng/giảm tốn tối đa `32` bước, lặp `|b|` lần ⇒ **Time `O(|b| · 32)`** — với `|b| <= 1000` là `~32 000` bước, vẫn ổn; nhưng nếu `b` có thể tới `2^31` thì **quá chậm**. **Space `O(1)`**.

**Nút thắt:** ta cộng từng đơn vị một trong khi `b` có thể rất lớn. Ta muốn xử lý **mọi bit của `b` cùng lúc** (tối đa `32` bước).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá: tách tổng thành "tổng không nhớ" và "nhớ", rồi lặp.**

`a + b = (a ^ b) + ((a & b) << 1)`.

Lý do: trên **mỗi cột bit**, `x + y = (x ^ y) + 2·(x & y)` (mục 3.2); nhân `2` cho phần nhớ chính là dịch trái một bit. Cộng trên mọi cột ⇒ đẳng thức trên đúng với cả số, không chỉ với bit.

Vế phải vẫn là **một phép cộng** (giữa `a ^ b` và phần nhớ), nhưng phần nhớ **đã bị đẩy sang trái ít nhất một bit**. Lặp lại cùng phép biến đổi: `(a, b) <- (a ^ b, (a & b) << 1)` cho đến khi **không còn nhớ** (`b == 0`) thì `a` là đáp án.

Với số **32 bit**, nhớ bị đẩy sang trái tối đa `32` lần là **rơi ra ngoài** (và bị cắt bởi mask), nên vòng lặp dừng sau **tối đa 32 bước**. Trong Python phải tự **cắt mask** ở mỗi bước (nếu không, số âm khiến vòng lặp chạy mãi — xem lỗi 1) và **đổi ngược** kết quả về số có dấu ở cuối.

In [9]:
# Đẳng thức a + b == (a ^ b) + ((a & b) << 1) trên nhiều cặp (chỉ để kiểm tra ý tưởng, dùng + cho vui)
for a, b in [(5, 3), (7, 7), (12, 5), (0, 9), (100, 28)]:
    left = a + b
    right = (a ^ b) + ((a & b) << 1)
    print(a, b, left, right, left == right)

5 3 8 8 True
7 7 14 14 True
12 5 17 17 True
0 9 9 9 True
100 28 128 128 True


## 7. Các cách tiếp theo

Ngoài Cách 1 (tăng/giảm từng đơn vị), ta có thêm **3 cách khác nhau về ý tưởng**: Cách 2 (full adder từng bit), Cách 3 (XOR + AND dịch trái, cách nộp), Cách 4 (tách dấu, cộng/trừ độ lớn).

### Cách 2 — Cộng từng bit với nhớ (full adder, "ripple carry")

**Giải thích từng đoạn.**

1. Duyệt `i = 0..31` (bit thấp đến cao). Lấy bit thứ `i` của `a` và `b`: `(a >> i) & 1` (cũng đúng với số âm vì dịch phải nhân bản bit dấu).
2. **Bit tổng** của cột: `bit_a ^ bit_b ^ carry`.
3. **Nhớ mới:** `1` nếu có **ít nhất hai** trong ba bit `bit_a`, `bit_b`, `carry` là `1`: `(bit_a & bit_b) | (carry & (bit_a ^ bit_b))`.
4. Ghép bit tổng vào `result` ở vị trí `i`: `result |= bit_sum << i`.
5. Cuối cùng `to_signed(result)` — sau 32 cột, mọi nhớ thừa bị bỏ (đúng quy tắc 32 bit).

Đây chính là **mạch cộng đủ (full adder)** nối tiếp trong CPU; cách làm giống hệt cộng tay trên giấy.

In [10]:
def get_sum_full_adder(a, b):
    result = 0
    carry = 0
    for i in range(32):
        bit_a = (a >> i) & 1
        bit_b = (b >> i) & 1
        bit_sum = bit_a ^ bit_b ^ carry
        carry = (bit_a & bit_b) | (carry & (bit_a ^ bit_b))
        result |= bit_sum << i
    return to_signed(result)


for a, b, expected in examples:
    print(get_sum_full_adder(a, b), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )
0 (kỳ vọng 0 )
-12 (kỳ vọng -12 )
0 (kỳ vọng 0 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionFullAdder:
    def getSum(self, a: int, b: int) -> int:
        mask = 0xFFFFFFFF
        result = carry = 0
        for i in range(32):
            x = (a >> i) & 1
            y = (b >> i) & 1
            result |= (x ^ y ^ carry) << i
            carry = (x & y) | (carry & (x ^ y))
        result &= mask
        return result if result <= 0x7FFFFFFF else ~(result ^ mask)


print(SolutionFullAdder().getSum(-2, 3), SolutionFullAdder().getSum(5, 3))     # 1 8

1 8


**Độ phức tạp Cách 2:** đúng `32` vòng lặp, mỗi vòng hằng số ⇒ **Time `O(32) = O(1)`**, **Space `O(1)`**.

### Cách 3 — XOR + AND dịch trái, lặp đến khi hết nhớ (cách nộp)

**Giải thích từng đoạn.**

1. `a &= mask; b &= mask`: đưa cả hai về dạng **32 bit không dấu** (số âm thành số lớn).
2. Lặp `while b:` — `b` giữ **phần nhớ** còn phải cộng.
3. `carry = ((a & b) << 1) & mask` — nhớ mới, cắt về 32 bit.
4. `a = a ^ b` — tổng không nhớ (không cần cắt thêm vì cả hai đều trong 32 bit).
5. `b = carry` — chuyển sang vòng sau.
6. Kết thúc: `a` là 32 bit. Nếu `a <= 0x7FFFFFFF` thì trả `a`; ngược lại là số âm: trả `~(a ^ mask)`.

In [12]:
def get_sum_xor_carry(a, b):
    mask = 0xFFFFFFFF
    a &= mask
    b &= mask
    while b:
        carry = ((a & b) << 1) & mask      # bit nhớ, dịch sang cột trái, giữ trong 32 bit
        a = a ^ b                          # tổng không nhớ
        b = carry                          # phần nhớ còn phải cộng
    return a if a <= 0x7FFFFFFF else ~(a ^ mask)


for a, b, expected in examples:
    print(get_sum_xor_carry(a, b), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )
0 (kỳ vọng 0 )
-12 (kỳ vọng -12 )
0 (kỳ vọng 0 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionXorCarry:
    def getSum(self, a: int, b: int) -> int:
        mask = 0xFFFFFFFF
        a &= mask
        b &= mask
        while b:
            a, b = a ^ b, ((a & b) << 1) & mask
        return a if a <= 0x7FFFFFFF else ~(a ^ mask)


print(SolutionXorCarry().getSum(-5, -7))     # -12

-12


**Độ phức tạp Cách 3:** tối đa `32` vòng ⇒ **Time `O(32) = O(1)`**, **Space `O(1)`**.

### Cách 4 — Tách dấu: cộng hoặc trừ **độ lớn**, không cần mask

**Giải thích từng đoạn.** Mask chỉ cần vì số âm của Python có vô hạn bit `1`. Nếu ta **chỉ làm việc với số không âm** thì mọi vòng lặp bit của Python tự dừng, **không cần mask**.

1. Tách `a` và `b` thành **dấu** (`a < 0`) và **độ lớn** (`abs(a)`).
2. **Cùng dấu:** cộng độ lớn bằng `add_nonneg` (`x, y = x ^ y, (x & y) << 1` đến khi `y = 0`), kết quả mang dấu chung.
3. **Khác dấu:** lấy số có độ lớn lớn hơn trừ số có độ lớn nhỏ hơn bằng `sub_nonneg` (mượn bit): `x, y = x ^ y, ((~x) & y) << 1`. Dấu kết quả là dấu của số có độ lớn **lớn hơn**.
4. **Đổi dấu không dùng `-`:** với `r >= 1`, `-r == ~(r - 1)`; ta tính `r - 1` bằng chính `sub_nonneg(r, 1)`.

**Vì sao `sub_nonneg` đúng?** Trên mỗi cột bit, `x - y` có bit hiệu là `x ^ y` và cần **mượn** từ cột trái khi `x = 0, y = 1`, tức `(~x) & y`; phần mượn dịch trái một bit. Ví dụ `5 - 3`: `101 ^ 011 = 110`, mượn `(~101 & 011) << 1 = 010 << 1 = 100`; lặp: `110 ^ 100 = 010`, mượn `(~110 & 100) << 1 = 0` ⇒ `010 = 2`.

In [14]:
def add_nonneg(x, y):
    """x + y với x, y >= 0, chỉ dùng bit (không cần mask vì số không âm)."""
    while y:
        x, y = x ^ y, (x & y) << 1
    return x


def sub_nonneg(x, y):
    """x - y với x >= y >= 0: bit hiệu là x ^ y, bit mượn là (~x & y) << 1."""
    while y:
        x, y = x ^ y, ((~x) & y) << 1
    return x


def negate(r):
    """-r không dùng dấu trừ: -r == ~(r - 1) với r >= 1."""
    return ~sub_nonneg(r, 1) if r else 0


def get_sum_sign(a, b):
    neg_a, neg_b = a < 0, b < 0
    x, y = abs(a), abs(b)
    if neg_a == neg_b:                      # cùng dấu: cộng độ lớn
        r = add_nonneg(x, y)
        return negate(r) if neg_a else r
    if x >= y:                              # khác dấu: lấy lớn trừ nhỏ, dấu theo số có độ lớn lớn hơn
        r, result_negative = sub_nonneg(x, y), neg_a
    else:
        r, result_negative = sub_nonneg(y, x), neg_b
    return negate(r) if result_negative else r


print(add_nonneg(5, 3), sub_nonneg(5, 3), negate(5), negate(0))     # 8 2 -5 0
for a, b, expected in examples:
    print(get_sum_sign(a, b), "(kỳ vọng", expected, ")")

8 2 -5 0
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )
0 (kỳ vọng 0 )
-12 (kỳ vọng -12 )
0 (kỳ vọng 0 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionSign:
    def getSum(self, a: int, b: int) -> int:
        def add(x, y):
            while y:
                x, y = x ^ y, (x & y) << 1
            return x

        def sub(x, y):                         # x >= y >= 0
            while y:
                x, y = x ^ y, ((~x) & y) << 1
            return x

        def neg(r):
            return ~sub(r, 1) if r else 0

        na, nb = a < 0, b < 0
        x, y = abs(a), abs(b)
        if na == nb:
            r = add(x, y)
            return neg(r) if na else r
        if x >= y:
            r, negative = sub(x, y), na
        else:
            r, negative = sub(y, x), nb
        return neg(r) if negative else r


print(SolutionSign().getSum(-1000, 1000), SolutionSign().getSum(-3, 8))      # 0 5

0 5


**Độ phức tạp Cách 4:** mỗi vòng lặp chạy tối đa bằng số bit của độ lớn (`~11` bit với `|x| <= 1000`; tổng quát `O(w)`) ⇒ **Time `O(w)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace **Cách 3** trên hai ví dụ: `a = 5, b = 3` (nhớ đẩy 3 lần) và `a = -2, b = 3` (số âm, nhớ lan qua nhiều bit). Để bảng dễ đọc, ta in 32 bit gom nhóm 4 bit bằng `format(x, "032_b")` (dấu `_` chỉ để ngăn cách).

Bảng trace Cách 3 với `a = 5`, `b = 3` (chỉ nêu 4 bit thấp; các bit cao đều `0`):

| Vòng | `a` (trước) | `b` (trước) | `a ^ b` | `(a & b) << 1` |
|---|---|---|---|---|
| 1 | `0101` | `0011` | `0110` | `0010` |
| 2 | `0110` | `0010` | `0100` | `0100` |
| 3 | `0100` | `0100` | `0000` | `1000` |
| 4 | `0000` | `1000` | `1000` | `0000` |

Kết thúc: `b = 0`, `a = 1000 = 8`. Với số âm, bit `1` bên trái (phần mở rộng dấu) khiến nhớ **chạy dọc** lên tận bit 31 rồi rơi ra ngoài — đó là lý do có thể cần tới `32` vòng (con số tối đa).

In [16]:
def trace_get_sum(a, b):
    mask = 0xFFFFFFFF
    print(f"a = {a}, b = {b}")
    a &= mask
    b &= mask
    step = 0
    while b:
        step += 1
        carry = ((a & b) << 1) & mask
        new_a = a ^ b
        print(f"vòng {step:2d}: a={format(a, '032_b')} b={format(b, '032_b')}")
        print(f"         a^b={format(new_a, '032_b')} carry={format(carry, '032_b')}")
        a, b = new_a, carry
    result = a if a <= 0x7FFFFFFF else ~(a ^ mask)
    print(f"kết thúc sau {step} vòng, a = {format(a, '032_b')} -> {result}")
    return result


assert trace_get_sum(5, 3) == 8

a = 5, b = 3
vòng  1: a=00_0000_0000_0000_0000_0000_0101 b=00_0000_0000_0000_0000_0000_0011
         a^b=00_0000_0000_0000_0000_0000_0110 carry=00_0000_0000_0000_0000_0000_0010
vòng  2: a=00_0000_0000_0000_0000_0000_0110 b=00_0000_0000_0000_0000_0000_0010
         a^b=00_0000_0000_0000_0000_0000_0100 carry=00_0000_0000_0000_0000_0000_0100
vòng  3: a=00_0000_0000_0000_0000_0000_0100 b=00_0000_0000_0000_0000_0000_0100
         a^b=00_0000_0000_0000_0000_0000_0000 carry=00_0000_0000_0000_0000_0000_1000
vòng  4: a=00_0000_0000_0000_0000_0000_0000 b=00_0000_0000_0000_0000_0000_1000
         a^b=00_0000_0000_0000_0000_0000_1000 carry=00_0000_0000_0000_0000_0000_0000
kết thúc sau 4 vòng, a = 00_0000_0000_0000_0000_0000_1000 -> 8


In [17]:
# Số âm: nhớ lan qua nhiều bit, in gọn lại (chỉ hiển thị 8 bit thấp mỗi vòng)
def trace_short(a, b):
    mask = 0xFFFFFFFF
    a &= mask
    b &= mask
    step = 0
    while b:
        step += 1
        a, b = a ^ b, ((a & b) << 1) & mask
        if step <= 8:                      # chỉ in 8 vòng đầu cho gọn
            print(f"vòng {step:2d}: 8 bit thấp a={format(a & 255, '08b')} b={format(b & 255, '08b')}")
    return a if a <= 0x7FFFFFFF else ~(a ^ mask), step


result, steps = trace_short(-2, 3)
print("kết quả:", result, "| số vòng:", steps)
assert result == 1

vòng  1: 8 bit thấp a=11111101 b=00000100
vòng  2: 8 bit thấp a=11111001 b=00001000
vòng  3: 8 bit thấp a=11110001 b=00010000
vòng  4: 8 bit thấp a=11100001 b=00100000
vòng  5: 8 bit thấp a=11000001 b=01000000
vòng  6: 8 bit thấp a=10000001 b=10000000
vòng  7: 8 bit thấp a=00000001 b=00000000
vòng  8: 8 bit thấp a=00000001 b=00000000
kết quả: 1 | số vòng: 31


Hãy nhìn: với `-2 + 3` ta mất `31` vòng (kết quả in ở cuối) vì phần nhớ lan dần từ bit `2` lên bit `31` rồi rơi ra khỏi 32 bit; bảng chỉ hiện 8 bit thấp nên từ vòng 7 trở đi cột `b` trông như toàn `0` (nhớ đã nằm ở các bit cao hơn). Khi cả hai số dương, thường dừng sớm.

In [18]:
def trace_sign(a, b):
    neg_a, neg_b = a < 0, b < 0
    x, y = abs(a), abs(b)
    print(f"a={a} (âm? {neg_a}, độ lớn {x}), b={b} (âm? {neg_b}, độ lớn {y})")
    if neg_a == neg_b:
        print("  cùng dấu -> cộng độ lớn:", x, "+", y)
        r = add_nonneg(x, y)
        result = negate(r) if neg_a else r
    else:
        big, small = (x, y) if x >= y else (y, x)
        negative = neg_a if x >= y else neg_b
        print(f"  khác dấu -> lấy {big} - {small}; kết quả {'âm' if negative else 'dương'}")
        r = sub_nonneg(big, small)
        result = negate(r) if negative else r
    print("  kết quả:", result)
    return result


assert trace_sign(-5, -7) == -12
assert trace_sign(-3, 8) == 5
assert trace_sign(4, -9) == -5

a=-5 (âm? True, độ lớn 5), b=-7 (âm? True, độ lớn 7)
  cùng dấu -> cộng độ lớn: 5 + 7
  kết quả: -12
a=-3 (âm? True, độ lớn 3), b=8 (âm? False, độ lớn 8)
  khác dấu -> lấy 8 - 3; kết quả dương
  kết quả: 5
a=4 (âm? False, độ lớn 4), b=-9 (âm? True, độ lớn 9)
  khác dấu -> lấy 9 - 4; kết quả âm
  kết quả: -5


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (XOR + AND dịch trái với mask 32 bit): ngắn, tối đa 32 vòng lặp, là cách chuẩn mà hầu hết lời giải Python dùng.

In [19]:
class Solution:
    def getSum(self, a: int, b: int) -> int:
        mask = 0xFFFFFFFF                       # mô phỏng số nguyên 32 bit
        a &= mask
        b &= mask
        while b:                                # b là phần nhớ còn phải cộng
            carry = ((a & b) << 1) & mask       # các cột sinh nhớ, dịch sang trái
            a = a ^ b                           # tổng không nhớ
            b = carry
        return a if a <= 0x7FFFFFFF else ~(a ^ mask)   # đổi 32 bit về số có dấu


sol = Solution()
for a, b, expected in examples:
    print(a, b, "->", sol.getSum(a, b), "(kỳ vọng", expected, ")")

1 2 -> 3 (kỳ vọng 3 )
2 3 -> 5 (kỳ vọng 5 )
-1 1 -> 0 (kỳ vọng 0 )
-5 -7 -> -12 (kỳ vọng -12 )
1000 -1000 -> 0 (kỳ vọng 0 )


## 10. Vì sao thuật toán đúng

- **Bất biến.** Sau mỗi vòng, `a + b` (tính **modulo `2^32`**) **không đổi**. Thật vậy, trên mỗi cột `x + y = (x ^ y) + 2·(x & y)`, nên `a + b = (a ^ b) + ((a & b) << 1)`; việc cắt mask chỉ bỏ các bội của `2^32`, không đổi kết quả modulo `2^32`.
- **Dừng.** Mỗi vòng, nhớ `b` mới là `(a & b) << 1`: **bit `1` thấp nhất của nó nằm ở vị trí cao hơn ít nhất 1** so với bit thấp nhất của `b` cũ (bit thấp nhất của một số khác 0 chỉ có thể dịch trái). Sau tối đa `32` vòng, `b` thành `0` do rơi khỏi 32 bit.
- **Kết quả đúng.** Khi `b = 0`, `a` là tổng modulo `2^32`, tức một số trong `[0, 2^32)`. Số nguyên 32 bit có dấu `s` được biểu diễn bởi `s mod 2^32` (bù hai). Ta đổi ngược bằng `~(a ^ mask)` khi `a > 2^31 - 1`, cho `a - 2^32`.
- **Tổng thật nằm trong khoảng 32 bit.** Với `|a|, |b| <= 1000`, tổng trong `[-2000, 2000]`, nên nằm trọn trong khoảng số 32 bit có dấu: kết quả modulo `2^32` đổi ngược chính xác bằng tổng thật.
- **Cách 4** đúng vì làm việc trên số không âm (cộng bằng XOR/AND, trừ bằng XOR/mượn) rồi gắn lại dấu theo độ lớn.

## 11. Phân tích Time / Space Complexity

Gọi `w = 32` là số bit.

| Cách | Time | Space |
|---|---|---|
| 1. Tăng/giảm từng đơn vị | `O(\|b\| · w)` | `O(1)` |
| 2. Full adder từng bit | `O(w)` | `O(1)` |
| 3. XOR + AND dịch trái (mask) | `O(w)` | `O(1)` |
| 4. Tách dấu, cộng/trừ độ lớn | `O(w)` | `O(1)` |

Vì `w = 32` là hằng số nên Cách 2, 3, 4 đều là `O(1)` theo nghĩa chặt. Cách 1 phụ thuộc **giá trị** `b` (không phải độ dài bit) nên chỉ dùng được vì `|b| <= 1000`.

## 12. Bộ test

Mọi cách trả về **một số duy nhất** nên so bằng `==`. **Oracle độc lập:** `a + b` của Python (ta được phép dùng `+` **trong test**, vì cấm chỉ áp dụng cho lời giải). Ngoài ra ta kiểm tra bằng `ast` rằng **code của lời giải không chứa `+` và `-`**.

In [20]:
sol = Solution()
impls = {
    "count": get_sum_count,
    "full_adder": get_sum_full_adder,
    "xor_carry": get_sum_xor_carry,
    "sign": get_sum_sign,
    "leetcode": sol.getSum,
}


def check_all(a, b, expected, label):
    for key, fn in impls.items():
        actual = fn(a, b)
        assert actual == expected, (label, key, a, b, actual, expected)
    print("PASS", label, f"({a}, {b}) -> {expected}")


for idx, (a, b, expected) in enumerate(examples, 1):
    check_all(a, b, expected, f"ví dụ {idx}")

PASS ví dụ 1 (1, 2) -> 3
PASS ví dụ 2 (2, 3) -> 5
PASS ví dụ 3 (-1, 1) -> 0
PASS ví dụ 4 (-5, -7) -> -12
PASS ví dụ 5 (1000, -1000) -> 0


In [21]:
# Kiểm tra bằng ast: các hàm lời giải không dùng toán tử + và - (kể cả += và -=, và dấu âm đơn)
import ast
import inspect

banned = (ast.Add, ast.Sub, ast.USub, ast.UAdd)
import textwrap
for obj in [Solution.getSum, SolutionFullAdder.getSum, SolutionXorCarry.getSum, SolutionSign.getSum,
            get_sum_full_adder, get_sum_xor_carry, get_sum_sign, add_nonneg, sub_nonneg, negate]:
    tree = ast.parse(textwrap.dedent(inspect.getsource(obj)))
    bad = [type(n.op).__name__ if hasattr(n, "op") else type(n).__name__
           for n in ast.walk(tree) if isinstance(getattr(n, "op", n), banned)]
    assert not bad, (obj.__qualname__, bad)
    print("không có + hoặc -:", obj.__qualname__)

không có + hoặc -: Solution.getSum
không có + hoặc -: SolutionFullAdder.getSum
không có + hoặc -: SolutionXorCarry.getSum
không có + hoặc -: SolutionSign.getSum
không có + hoặc -: get_sum_full_adder
không có + hoặc -: get_sum_xor_carry
không có + hoặc -: get_sum_sign
không có + hoặc -: add_nonneg
không có + hoặc -: sub_nonneg
không có + hoặc -: negate


### 12.1. Các trường hợp biên

In [22]:
check_all(0, 0, 0, "hai số 0")
check_all(0, 7, 7, "a = 0")
check_all(-7, 0, -7, "b = 0, a âm")
check_all(1, -1, 0, "triệt tiêu")
check_all(-1, -1, -2, "hai số -1")
check_all(1000, 1000, 2000, "biên trên")
check_all(-1000, -1000, -2000, "biên dưới")
check_all(1000, -1000, 0, "đối nhau ở biên")
check_all(-1000, 1000, 0, "đối nhau ở biên (đảo)")
check_all(999, 1, 1000, "tràn nhớ dài (1111100111 + 1)")
check_all(511, 1, 512, "nhớ lan qua nhiều bit 1")
check_all(-512, 511, -1, "âm cộng dương ra -1")
check_all(-1, 1, 0, "-1 + 1")
check_all(-2, 3, 1, "ví dụ trace")
check_all(7, -3, 4, "dương lớn cộng âm nhỏ")
check_all(3, -7, -4, "dương nhỏ cộng âm lớn")

PASS hai số 0 (0, 0) -> 0
PASS a = 0 (0, 7) -> 7
PASS b = 0, a âm (-7, 0) -> -7
PASS triệt tiêu (1, -1) -> 0
PASS hai số -1 (-1, -1) -> -2
PASS biên trên (1000, 1000) -> 2000
PASS biên dưới (-1000, -1000) -> -2000
PASS đối nhau ở biên (1000, -1000) -> 0
PASS đối nhau ở biên (đảo) (-1000, 1000) -> 0
PASS tràn nhớ dài (1111100111 + 1) (999, 1) -> 1000
PASS nhớ lan qua nhiều bit 1 (511, 1) -> 512
PASS âm cộng dương ra -1 (-512, 511) -> -1
PASS -1 + 1 (-1, 1) -> 0
PASS ví dụ trace (-2, 3) -> 1
PASS dương lớn cộng âm nhỏ (7, -3) -> 4
PASS dương nhỏ cộng âm lớn (3, -7) -> -4


### 12.2. Phản ví dụ cho các cách làm sai

Các lỗi hay gặp: (a) **chỉ `a ^ b`** (quên nhớ); (b) **quên đổi về số có dấu** (trả `4294967295` thay vì `-1`); (c) **thứ tự ưu tiên toán tử**: `a & b << 1` khác `(a & b) << 1`. Ta chứng minh các hàm sai này khác đúng ở một số input.

In [23]:
def wrong_xor_only(a, b):
    return a ^ b                                  # SAI: thiếu phần nhớ


def wrong_no_signed_convert(a, b):
    mask = 0xFFFFFFFF
    a &= mask
    b &= mask
    while b:
        a, b = a ^ b, ((a & b) << 1) & mask
    return a                                      # SAI: không đổi về số có dấu


def wrong_precedence(a, b):
    mask = 0xFFFFFFFF
    a &= mask
    b &= mask
    for _ in range(40):                           # giới hạn vòng để không treo
        if not b:
            break
        a, b = a ^ b, (a & b << 1) & mask         # SAI: a & (b << 1), không phải (a & b) << 1
    return a if a <= 0x7FFFFFFF else ~(a ^ mask)


print("xor only (5, 3):", wrong_xor_only(5, 3), "!=", sol.getSum(5, 3))
print("no signed convert (-1, 0):", wrong_no_signed_convert(-1, 0), "!=", sol.getSum(-1, 0))
print("precedence (5, 3):", wrong_precedence(5, 3), "!=", sol.getSum(5, 3))
assert wrong_xor_only(5, 3) != sol.getSum(5, 3)
assert wrong_no_signed_convert(-1, 0) != sol.getSum(-1, 0)
assert wrong_precedence(5, 3) != sol.getSum(5, 3)

xor only (5, 3): 6 != 8
no signed convert (-1, 0): 4294967295 != -1
precedence (5, 3): 2 != 8


### 12.3. Đối chiếu vét cạn và ngẫu nhiên (có seed)

Ràng buộc đề: `-1000 <= a, b <= 1000`. Ta kiểm tra:

1. **Vét cạn** mọi cặp trong `[-40, 40]` cho **cả 5 hàm** (Cách 1 chậm hơn nên khoảng nhỏ).
2. **Vét cạn thưa** trên toàn khoảng: `a` từ `-1000` đến `1000` bước `7`, `b` **mọi** giá trị — cho Cách 2, 3, 4 và bản nộp.
3. **Ngẫu nhiên có seed** trên toàn khoảng đề, tất cả 5 hàm.
4. **Ngoài ràng buộc** (số 32 bit, tổng chưa tràn): chỉ kiểm tra Cách 2, 3 (Cách 1 quá chậm, Cách 4 vẫn đúng).

In [24]:
for a in range(-40, 41):
    for b in range(-40, 41):
        ref = a + b                                   # oracle (được dùng + trong test)
        for key, fn in impls.items():
            assert fn(a, b) == ref, (key, a, b)
print("PASS vét cạn 81 x 81 cặp, 5 hàm")

fast = {k: v for k, v in impls.items() if k != "count"}
for a in range(-1000, 1001, 7):
    for b in range(-1000, 1001):
        ref = a + b
        for key, fn in fast.items():
            assert fn(a, b) == ref, (key, a, b)
print("PASS vét cạn thưa: 286 giá trị a x 2001 giá trị b, 4 hàm")

rng = random.Random(371)
for trial in range(3000):
    a, b = rng.randint(-1000, 1000), rng.randint(-1000, 1000)
    ref = a + b
    for key, fn in impls.items():
        assert fn(a, b) == ref, (key, a, b)
print("PASS 3000 cặp ngẫu nhiên đúng ràng buộc")

for trial in range(3000):
    a, b = rng.randint(-2 ** 30, 2 ** 30 - 1), rng.randint(-2 ** 30, 2 ** 30 - 1)   # tổng nằm trong 32 bit có dấu
    ref = a + b
    for key in ("full_adder", "xor_carry", "sign", "leetcode"):
        assert impls[key](a, b) == ref, (key, a, b)
print("PASS 3000 cặp ngoài ràng buộc (số 32 bit)")

PASS vét cạn 81 x 81 cặp, 5 hàm


PASS vét cạn thưa: 286 giá trị a x 2001 giá trị b, 4 hàm


PASS 3000 cặp ngẫu nhiên đúng ràng buộc
PASS 3000 cặp ngoài ràng buộc (số 32 bit)


Vì sao lại kiểm tra thêm **ngoài ràng buộc**? Vì giải pháp mô phỏng 32 bit có dấu, nên đúng với **mọi** `a, b` mà tổng còn nằm trong `[-2^31, 2^31 - 1]`. (Nếu tổng tràn, kết quả sẽ "quay vòng" giống C++/Java — hành vi này không xảy ra với ràng buộc của đề.)

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên mask — vòng lặp vô hạn khi có số âm

In [25]:
def loi1_khong_mask(a, b, limit=100):
    steps = 0
    while b:
        a, b = a ^ b, (a & b) << 1        # SAI: không cắt về 32 bit
        steps += 1
        if steps >= limit:
            return None, steps            # thoát khẩn để notebook không treo
    return a, steps


print("5 + 3  :", loi1_khong_mask(5, 3))          # số dương vẫn chạy đúng
print("-1 + 1 :", loi1_khong_mask(-1, 1))         # số âm: không bao giờ hết nhớ
print("đúng   :", Solution().getSum(-1, 1))

5 + 3  : (8, 4)
-1 + 1 : (None, 100)
đúng   : 0


**Vì sao sai:** số âm trong Python có vô hạn bit `1` bên trái, nên nhớ `(a & b) << 1` cứ **trôi mãi sang trái** mà không bao giờ rơi ra ngoài (không có giới hạn 32 bit như C++). **Cách sửa:** luôn `& mask` (`0xFFFFFFFF`) để mô phỏng 32 bit, hoặc dùng Cách 4 (làm trên số không âm).

### Lỗi 2: Không đổi kết quả về số có dấu

In [26]:
print("sai:", wrong_no_signed_convert(-3, 1), "| đúng:", Solution().getSum(-3, 1))
print("sai:", wrong_no_signed_convert(-1, 0), "| đúng:", Solution().getSum(-1, 0))

sai: 4294967294 | đúng: -2
sai: 4294967295 | đúng: -1


**Vì sao sai:** sau khi cắt mask, `-2` được biểu diễn thành `4294967294` (32 bit). Nếu trả thẳng, ta nhận số khổng lồ. **Cách sửa:** `return a if a <= 0x7FFFFFFF else ~(a ^ mask)`.

### Lỗi 3: Đổi ngược sai công thức

In [27]:
def loi3_doi_nguoc_sai(a, b):
    mask = 0xFFFFFFFF
    a &= mask
    b &= mask
    while b:
        a, b = a ^ b, ((a & b) << 1) & mask
    return a if a <= 0x7FFFFFFF else ~a        # SAI: ~a chưa đủ, a vẫn đang là số 32 bit không dấu


print("sai:", loi3_doi_nguoc_sai(-3, 1), "| đúng:", Solution().getSum(-3, 1))
print("~a khi a = 4294967294:", ~4294967294, "| ~(a ^ mask):", ~(4294967294 ^ 0xFFFFFFFF))

sai: -4294967295 | đúng: -2
~a khi a = 4294967294: -4294967295 | ~(a ^ mask): -2


**Vì sao sai:** `~a` với `a = 4294967294` cho `-4294967295` (phép đảo bit **vô hạn** của Python). Phải **đảo 32 bit thấp trước** (`a ^ mask`) rồi mới đảo `~` để đưa về dạng "vô hạn bit 1" của số âm. **Cách sửa:** `~(a ^ mask)`.

### Lỗi 4: Nhầm thứ tự ưu tiên toán tử

In [28]:
a, b = 5, 3
print("a & b << 1   =", a & b << 1, "  (= a & (b << 1))")
print("(a & b) << 1 =", (a & b) << 1)

a & b << 1   = 4   (= a & (b << 1))
(a & b) << 1 = 2


**Vì sao sai:** trong Python, `<<` có **độ ưu tiên cao hơn** `&`, nên `a & b << 1` được hiểu là `a & (b << 1)`. Ở trên `5 & 6 = 4`, trong khi nhớ đúng là `(5 & 3) << 1 = 2`. **Cách sửa:** luôn **thêm ngoặc**: `((a & b) << 1)`.

### Lỗi 5: Dùng `+` (hoặc `sum`) — vi phạm đề

In [29]:
def loi5_dung_cong(a, b):
    return a + b                                   # SAI về mặt luật: đề cấm + và -


print(loi5_dung_cong(1, 2), "(đúng giá trị nhưng bị loại / không được điểm khi người ra đề kiểm tra)")

3 (đúng giá trị nhưng bị loại / không được điểm khi người ra đề kiểm tra)


**Vì sao sai:** đề yêu cầu **không dùng `+`, `-`**; `sum([a, b])`, `operator.add`, `a.__add__(b)` đều là "lách luật". Mục đích bài là hiểu **mạch cộng bằng bit**. **Cách sửa:** dùng `^`, `&`, `<<` như các cách trên. Chúng ta chỉ dùng `+` trong **test** để làm oracle.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Cộng/trừ/nhân/chia mà cấm dùng toán tử số học**" ⇒ nghĩ ngay tới **bit**: XOR là cộng không nhớ, AND dịch trái là nhớ.
- Vòng lặp chuẩn: `while b: a, b = a ^ b, (a & b) << 1`. Ở Python **phải thêm mask 32 bit** và **đổi lại số có dấu** vì `int` không giới hạn.
- Công thức đổi ngược: `x if x <= 0x7FFFFFFF else ~(x ^ 0xFFFFFFFF)`.
- Phép trừ: `a - b = a + (~b + 1)` (cộng với **bù hai** của `b`), hay `a ^ b` với mượn `(~a & b) << 1`.
- Thứ tự ưu tiên trong Python (cao → thấp): `~`, `<<`/`>>`, `&`, `^`, `|`. **Luôn dùng ngoặc**.
- Có thể thử nhanh bằng `format(x & 0xFFFFFFFF, "032b")` để thấy 32 bit.
- Giới hạn của đề (`|a|, |b| <= 1000`) cho phép cả cách chậm (Cách 1), nhưng mục đích bài là cách `O(32)`.

## 15. Biến thể của bài

1. **Subtract Two Integers** — `a - b` mà không dùng `-`: `a + (~b + 1)`, nhưng tự cài `+` bằng bit.
2. **Multiply Strings (43)** / **Multiply without `*`** — "dịch và cộng" (Russian peasant): `a * b` bằng cộng `a << i` với mỗi bit `1` của `b`.
3. **Divide Two Integers (29)** — chia bằng cách trừ dần các bội `divisor << k`.
4. **Add Binary (67)**, **Add Strings (415)**, **Add to Array-Form of Integer (989)** — cùng ý "cộng có nhớ".
5. **Sum of Two Integers với 64 bit** — đổi mask thành `0xFFFFFFFFFFFFFFFF`.
6. **Single Number (136)**, **Counting Bits (338)** — các bài khác về XOR/AND bit.

In [30]:
def add_bits(a, b):
    return get_sum_xor_carry(a, b)


def subtract_bits(a, b):
    """a - b = a + (~b + 1), với + được thay bằng add_bits (và +1 cũng qua add_bits)."""
    return add_bits(a, add_bits(~b, 1))


def multiply_bits(a, b):
    """a * b bằng 'dịch và cộng' trên |b|, rồi gắn dấu."""
    negative = (a < 0) != (b < 0)
    x, y = abs(a), abs(b)
    result = 0
    while y:
        if y & 1:
            result = add_bits(result, x)
        x <<= 1
        y >>= 1
    return subtract_bits(0, result) if negative else result


def sum_64(a, b):
    mask = 0xFFFFFFFFFFFFFFFF
    a &= mask
    b &= mask
    while b:
        a, b = a ^ b, ((a & b) << 1) & mask
    return a if a <= 0x7FFFFFFFFFFFFFFF else ~(a ^ mask)


print(subtract_bits(10, 3), subtract_bits(3, 10), multiply_bits(7, -6), multiply_bits(-4, -5), sum_64(2 ** 40, -(2 ** 41)))
for a in range(-30, 31):
    for b in range(-30, 31):
        assert subtract_bits(a, b) == a - b
        assert multiply_bits(a, b) == a * b
print("subtract_bits và multiply_bits đúng trên [-30, 30]^2")

7 -7 -42 20 -1099511627776
subtract_bits và multiply_bits đúng trên [-30, 30]^2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tăng/giảm từng đơn vị | `inc`/`dec` bằng bit, lặp `\|b\|` lần | `O(\|b\|·w)` | `O(1)` | học cơ chế; chỉ khi `\|b\|` nhỏ |
| 2. Full adder | cộng từng cột bit với nhớ | `O(w)` | `O(1)` | hiểu mạch cộng thật của CPU |
| 3. XOR + AND dịch trái | `(a ^ b, (a & b) << 1)` đến khi hết nhớ | `O(w)` | `O(1)` | **nên nộp**: ngắn, chuẩn |
| 4. Tách dấu | cộng/trừ độ lớn, gắn dấu sau | `O(w)` | `O(1)` | tránh mask; làm việc với số không âm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 & 3`, `5 | 3`, `5 ^ 3`, `3 << 2`, `bin(10)`, `-1 & 0xFF`, và `Solution().getSum(4, 4)`.

**Bài 2 (tính tay).** Dùng cách `(a ^ b, (a & b) << 1)` để cộng `6 + 7` ở dạng 4 bit: lập bảng từng vòng (`a`, `b`, `a ^ b`, `(a & b) << 1`) cho tới khi `b = 0`.

**Bài 3 (điền TODO).** Hoàn thành `get_sum_todo(a, b)` theo Cách 3.

In [31]:
def get_sum_todo(a, b):
    # TODO 1: mask = 0xFFFFFFFF; a &= mask; b &= mask
    # TODO 2: while b: tính carry = ((a & b) << 1) & mask; a = a ^ b; b = carry
    # TODO 3: nếu a <= 0x7FFFFFFF thì trả a, ngược lại trả ~(a ^ mask)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `get_sum_recursive(a, b)` bằng **đệ quy**: `if b == 0: trả a`; ngược lại gọi lại với `(a ^ b, (a & b) << 1)` có mask, và đổi về số có dấu ở lớp ngoài cùng (gợi ý: viết hàm phụ làm việc với số 32 bit không dấu rồi bọc ngoài bằng `to_signed`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `negate_bits(x)` trả `-x` chỉ bằng bit và `+` tự cài (gợi ý: `~x` rồi cộng `1` bằng `add_bits`). (b) Viết `is_power_of_two(n)` (`n > 0`) chỉ bằng `&` và so sánh (gợi ý: `n & (n - 1)`, nhưng bạn tự thay `n - 1` bằng hàm bit nếu muốn thử thách). (c) Với `a, b` trong khoảng 8 bit có dấu (`-128..127`), sửa mask để mô phỏng số 8 bit: kết quả khi tràn (`100 + 100`) là gì?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `7`, `6`, `12`, `'0b1010'`, `255`, `8`. Bài 3: chép lại sáu dòng của Cách 3; nhớ thêm mask ở `carry`.
- **Mức 2 (vừa):** Bài 2: `0110, 0111 -> xor 0001, carry (0110&0111)<<1 = 1100`; `0001, 1100 -> xor 1101, carry 0`; kết quả `1101 = 13`. Bài 4: điều kiện dừng là `b == 0`; phần nhớ nhớ `& mask`.
- **Mức 3 (sâu):** Bài 5(a): `add_bits(~x, 1)`. 5(b): `n & (n - 1)` xóa bit `1` thấp nhất; lũy thừa của 2 chỉ có **một** bit `1`, nên kết quả bằng `0`. 5(c): với mask `0xFF` và ngưỡng `0x7F`, `100 + 100 = 200 = 0b11001000` được đọc là số âm `200 - 256 = -56` (tràn).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert (5 & 3, 5 | 3, 5 ^ 3, 3 << 2) == (1, 7, 6, 12)
assert bin(10) == "0b1010"
assert -1 & 0xFF == 255
assert Solution().getSum(4, 4) == 8

# Bài 2: 6 + 7 trong 4 bit
a, b, rows = 6, 7, []
while b:
    rows.append((a, b, a ^ b, (a & b) << 1))
    a, b = a ^ b, (a & b) << 1
assert rows == [(6, 7, 1, 12), (1, 12, 13, 0)]
assert a == 13


# Bài 3
def get_sum_todo_ans(a, b):
    mask = 0xFFFFFFFF
    a &= mask
    b &= mask
    while b:
        carry = ((a & b) << 1) & mask
        a = a ^ b
        b = carry
    return a if a <= 0x7FFFFFFF else ~(a ^ mask)


for a, b, expected in examples:
    assert get_sum_todo_ans(a, b) == expected


# Bài 4
def get_sum_recursive(a, b):
    mask = 0xFFFFFFFF

    def go(x, y):
        if y == 0:
            return x
        return go(x ^ y, ((x & y) << 1) & mask)

    return to_signed(go(a & mask, b & mask))


for a, b, expected in examples:
    assert get_sum_recursive(a, b) == expected
assert get_sum_recursive(-1000, -1000) == -2000


# Bài 5(a)
def negate_bits(x):
    return add_bits(~x, 1)


for x in range(-50, 51):
    assert negate_bits(x) == -x


# Bài 5(b)
def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0


assert [n for n in range(1, 40) if is_power_of_two(n)] == [1, 2, 4, 8, 16, 32]


# Bài 5(c)
def sum_8bit(a, b):
    mask = 0xFF
    a &= mask
    b &= mask
    while b:
        a, b = a ^ b, ((a & b) << 1) & mask
    return a if a <= 0x7F else ~(a ^ mask)


assert sum_8bit(100, 100) == -56
assert sum_8bit(-5, 3) == -2
print("Đáp án bài 1-5 đều đúng")

Đáp án bài 1-5 đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cộng hai số nguyên không dùng `+`/`-`, mô phỏng 32 bit có dấu trong Python)

```python
class Solution:
    def getSum(self, a: int, b: int) -> int:
        mask = 0xFFFFFFFF
        a &= mask
        b &= mask
        while b:
            a, b = a ^ b, ((a & b) << 1) & mask
        return a if a <= 0x7FFFFFFF else ~(a ^ mask)
```

**Checklist**

- [ ] `a` và `b` đều đã được `& mask` **trước** khi vào vòng lặp?
- [ ] Phần nhớ `((a & b) << 1)` có **ngoặc** và có `& mask`?
- [ ] Vòng lặp dừng khi `b == 0` (không còn nhớ)?
- [ ] Cuối cùng đã đổi về **số có dấu** bằng `~(a ^ mask)` khi `a > 0x7FFFFFFF`?
- [ ] Không có `+`, `-`, `sum`, `operator.add` trong lời giải?
- [ ] Đã thử `(-1, 1) -> 0`, `(-5, -7) -> -12`, `(1000, -1000) -> 0`, `(0, 0) -> 0`?